In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time
66
#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, 
                                          get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_day0,
                                         save_calibration_data_new_day)

from utils.calcium import calcium

#
print ("...DONE...")

Operating system:  Windows


Autosaving every 180 seconds


auto.py (22): IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


...DONE...


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

# load day2 onwards raw image
fname_day0 = r'F:\bmi\cohort12\DON-016550\day0\calibration\Image_001_001.raw'
fname_current_day = r'F:\bmi\cohort12\DON-016550\20230831\calibration\Image_001_001.raw'

# fname_day0 = r'F:\bmi\cohort12\DON-016551\day0\calibration\Image_001_001.raw'
# fname_current_day = r'F:\bmi\cohort12\DON-016551\20230830\calibration\Image_001_001.raw'

# fname_day0 = r'F:\bmi\cohort12\DON-015961\day0\calibration\Image_001_001.raw'
# fname_current_day = r'F:\bmi\cohort12\DON-015961\20230830\calibration\Image_001_001.raw'

# fname_day0 = r'F:\bmi\cohort12\DON-015962\day0\calibration\Image_001_001.raw'
# fname_current_day = r'F:\bmi\cohort12\DON-015962\20230831\calibration\Image_001_001.raw'


bmi_c = CalibrationTools(fname_current_day)
bmi_c.calcium = calcium
bmi_c.fname_day0 = fname_day0
#
#bmi_c.smooth_ca_time_series = smooth_ca_time_series
#bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

#
print ("DONE...")

memmap :  (25000, 512, 512)
DONE...


In [3]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
# load suite2p footprints
print (bmi_c.fname_day0)
bmi_c = get_footprints_from_day0(bmi_c)
print ("DONE...")

F:\bmi\cohort12\DON-016550\day0\calibration\Image_001_001.raw
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (335, 90000)
         self.Fneu (neuropile):  (335, 90000)
         self.iscell (Suite2p cell classifier output):  (383, 2)
              of which number of good cells:  (335,)
         self.spks (deconnvoved spikes):  (335, 90000)
         self.stat (footprints structure):  (335,)
         mean std over all cells :  58.5
# of footprints;  335
to load fname:  F:\bmi\cohort12\DON-016550\day0\rois_pixels_and_thresholds_day0.npz
ENSMBEL CELL IDS:  [11 25 20 58]
DONE...


In [4]:
#########################################################
########### GRAB CELLS  ##################
#########################################################
# #
bmi_c.scale=3  
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 1       # Subsample the time series to go faster;

# visualize traces
bmi_c.vmin=0
bmi_c.vmax=3000
bmi_c.max_n_cells = 50

#
order_type = 'snr'  # 'snr' or 'f0'
bmi_c.compute_roi_traces_f0_and_NO_reorder_cells_Day1()

#
bmi_c.visualize_traces_snr_order2(bmi_c.std_map)
print ("DONE...")
#

computing roi traces for SNR indexing: 100%|███████████████████████████████████████| 2500/2500 [00:18<00:00, 136.04it/s]


memmap :  (25000, 512, 512)
cell ids:  [11 25 20 58]
DONE...


In [5]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
bmi_c.show_traces_ids(bmi_c.both)
print ("DONE...")

all cells: [11 25 20 58]
DONE...


In [6]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles_new_day(bmi_c.std_map)

print ("DONE...")

100%|███████████████████████████████████████████████████████████████████████████| 25000/25000 [00:03<00:00, 7890.02it/s]


cell ids:  [11 25 20 58]
DONE...


In [7]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# if using binning
bmi_c.binning_flag = True
bmi_c.binning_time = 0.200
bmi_c.smoothing_n_bins = 3

# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 3   # reward lockout in seconds
bmi_c.post_missed_reward_lockout = 3
bmi_c.post_reward_lockout_baseline_min = .5 # want to wait until we drop to 30% of threshold
bmi_c.trial_time = 30
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.25 #*0.85

#gr.find_reward_thresholds_low_and_high()
#bmi_c.high = 2
stepper = 0.8
bmi_c.find_reward_thresholds_high_realtime(stepper)  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


# do not change this
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)
print ("DONE...")

nsec recording:  833 max # of random rewards (i.e. every 30sec)  27
 @0.25% reward rate:  6
 high guess:  1.5131689505347148
updated rewards #:  0  for threshold:  1.5131689505347148
updated rewards #:  0  for threshold:  1.2105351604277719
updated rewards #:  1  for threshold:  0.9684281283422176
updated rewards #:  5  for threshold:  0.7747425026737741
updated rewards #:  8  for threshold:  0.6197940021390194
updated rewards #:  5  for threshold:  0.7747425026737742
updated rewards #:  5  for threshold:  0.7669950776470364
updated rewards #:  5  for threshold:  0.7593251268705661
updated rewards #:  5  for threshold:  0.7517318756018604
updated rewards #:  6  for threshold:  0.7442145568458418
FINAL # of rewards #:  6 , set threshold:  0.7442145568458418
thresholds:  0.7442145568458418
 PROCESSING...
bmi_c.high: scaled by:  1.0 , final value:   0.7442145568458418
DONE...


In [8]:
#############################################
########### SAVE DATA #######################
#############################################

#
save_calibration_data_new_day(bmi_c)  

print ("DONE...")

 couldn't save bmi_c.object .... TO FIX!
Done...
DONE...


In [49]:
data = np.load('/media/cat/4TBSSD/donato/DON8498/22-06-08/rois_pixels_and_thresholds.npz',
               allow_pickle=True)

ensemble1_footprints = data['ensemble1_footprints']
contours_local = data['ensemble1_contours']

FileNotFoundError: [Errno 2] No such file or directory: '/media/cat/4TBSSD/donato/DON8498/22-06-08/rois_pixels_and_thresholds.npz'

In [4]:
cell_id =1
plt.figure()
temp = bmi_c.roi_traces[cell_id]
plt.plot((temp - np.median(temp))/np.median(temp))
plt.plot((temp - bmi_c.roi_f0s[cell_id])/bmi_c.roi_f0s[cell_id])
plt.show()

NameError: name 'bmi_c' is not defined

In [6]:
plt.figure()
for c in range(len(contours_local)):
    for k in range(len(contours_local[c])-1):

        plt.plot([contours_local[c][k][0],
                            contours_local[c][k+1][0]],
                           [contours_local[c][k][1],
                            contours_local[c][k+1][1]],
                            c='blue',
                            linewidth=5)
plt.show()


In [47]:
d = np.load('/media/cat/4TB/donato/andres/mic_test/audio_data.npy')
print ("d: ", d.shape)

t = np.load('/media/cat/4TB/donato/andres/mic_test/audio_data_timestamps.npy')
print (t)

tv = np.load('/media/cat/4TB/donato/andres/mic_test/video_data.npz')['vide_frame_times']
print (tv)

#
plt.figure()
plt.plot(t-t[0], t*0+0, label='audio')
plt.plot(tv[:,1]-t[0], tv[:,1]*0+1, label='video')
plt.legend()
plt.xlabel("sec")
plt.show()



d:  (2842,)
[1.67958427e+09 1.67958427e+09 1.67958427e+09 ... 1.67958430e+09
 1.67958430e+09 1.67958430e+09]
[[1.00000000e+00 1.67958427e+09]
 [2.00000000e+00 1.67958427e+09]
 [3.00000000e+00 1.67958427e+09]
 ...
 [9.98000000e+02 1.67958431e+09]
 [9.99000000e+02 1.67958431e+09]
 [1.00000000e+03 1.67958431e+09]]


In [16]:
data1 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_regular_filter.npz',
                allow_pickle=True)
e1 = data1['ensemble_activity']
print (e1.shape)


data2 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_5timesteps.npz',
                allow_pickle=True)
e2 = data2['ensemble_activity']
print (e2.shape)

data3 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_no_filter.npz',
                allow_pickle=True)
e3 = data3['ensemble_activity']
print (e3.shape)

data4 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results.npz',
                allow_pickle=True)
e4 = data4['ensemble_activity']
print (e4.shape)


#############################################################
plt.figure()

t = np.arange(e1.shape[1])/30
plt.plot(t,
         e1[0]-e1[1],
         c='blue', 
         linewidth = 4,
         label='current filter')
plt.plot(t,
         e2[0]-e2[1],
         linewidth = 4,
         c='red', label = 'mean last 5 time steps')
plt.plot(t,
         e3[0]-e3[1],
         linewidth = 4,
         c='green', label = 'no filter')
plt.plot(t,
         e4[0]-e4[1],
         linewidth = 4,
         c='black', label = 'graded filter last 5 time steps')


plt.xlabel("time (sec)", fontsize=16)
plt.ylabel("Ensemble 1 - Ensemble 2", fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.plot([t[0],t[-1]],
         [0,0],
         '--',
        c='black')
plt.legend()
plt.show()

(2, 3000)
(2, 3000)
(2, 3000)
(2, 3000)


In [3]:
def get_octave_frequencies(low_freq,
						   high_freq,
						   octave_size=0.25):
	#
	octaves = []

	#
	octaves.append(low_freq)
	temp = low_freq
	while True:
		temp = temp * (1 + octave_size)
		if temp > high_freq:
			break
		octaves.append(temp)
	"""
	low_freq = 1000
	high_freq = 16000
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
    
	octaves = 2**(octave_size*x)
    
	#
	return np.array(1000*octaves)
	"""
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
	octaves = 2**(octave_size*x)
	return np.array(1000*octaves)

	#

In [4]:
low_freq = 2000
high_freq = 16000

octaves = get_octave_frequencies(low_freq,high_freq,octave_size=0.25)

NameError: name 'x' is not defined